# Clustering

In [ ]:
import os
import warnings
import pandas as pd
import seaborn as sns

from kneed import KneeLocator
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

pd.set_option('display.max_columns', None)

In [ ]:
from utils.constants import RANDOM_SEED
from utils.common import get_data_folder_path, set_plot_font_sizes
from utils.clustering import search_kmeans, plot_kmeans_search, plot_cluster_boxplots

In [ ]:
# plots configuration
sns.set_style("darkgrid")
sns.set_palette("colorblind")
set_plot_font_sizes()
%matplotlib inline

## Preprocessing

### Load data

In [ ]:
data_path = get_data_folder_path()

df_input = pd.read_csv(os.path.join(data_path, 'fetal_health.csv'))
df_input.columns = [col.replace(' ', '_') for col in df_input.columns]

In [ ]:
# define columns for clustering
cluster_cols = [
    col for col in df_input.columns
    # remove the target column to simulate an unsupervised problem
    if col != "fetal_health"
    # remove all histogram-derived and variability features to simplify the clustering process
    and not col.startswith("histogram_")
    and not col.endswith("_variability")
]
df_cl = df_input[cluster_cols]

### Scale data (if necessary)

If all features used for clustering have the same range (e.g. scores form 0 to 100) or the same unit (e.g. distances), there is no need to standardize the data.

In [ ]:
# Standardize X_train and X_test
stdscaler = StandardScaler()
df_cl_std = pd.DataFrame(stdscaler.fit_transform(df_cl), columns=df_cl.columns, index=df_cl.index)

## K-means

### Find best number of clusters

In [ ]:
df_kmeans = search_kmeans(df_cl_std, max_n_clusters=15)

Elbow Method implementation:
- Kneedle algorithm original paper: https://www1.icsi.berkeley.edu/~barath/papers/kneedle-simplex11.pdf
- `kneed` python package: https://github.com/arvkevi/kneed

In [ ]:
# determine the ideal number of cluster using the "Elbow Method"
# using the kneed package which implements the Kneedle algorithm
kl = KneeLocator(
    x=df_kmeans["n_clusters"].values,
    y=df_kmeans["wcss"].values,
    curve="convex",
    direction="decreasing"
)
print(f'Elbow Method: best number of clusters is {kl.elbow}')

In [ ]:
fig = plot_kmeans_search(df_kmeans=df_kmeans, elbow=kl.elbow)

### Fit final model

In [ ]:
# fit K-means with selected number of clusters
kmeans_model = KMeans(n_clusters=kl.elbow, verbose=0, random_state=RANDOM_SEED)
kmeans_model.fit(df_cl_std)

In [ ]:
s_clusters = pd.Series(data=kmeans_model.labels_, name="cluster", index=df_cl_std.index)
s_clusters += 1  # set first cluster as 1 instead of 0

with warnings.catch_warnings(action="ignore"):
    df_cl_std.loc[:, "cluster"] = s_clusters
    df_cl.loc[:, 'cluster'] = s_clusters
    df_input.loc[:, "cluster"] = s_clusters

### Describe clusters

In [ ]:
fig = plot_cluster_boxplots(
    df=df_cl,
    cluster_col="cluster",
    plots_per_line=2,
    title="Features used in K-means Clustering",
)

In [ ]:
fig = plot_cluster_boxplots(
    df=df_input,
    cluster_col="cluster",
    plots_per_line=2,
    title="All features from input dataset",
)